In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os, sys

GOOGLE_DRIVE_PATH_POST_MYDRIVE = 'OMSCS/DeepLearning/main_repo'
GOOGLE_DRIVE_PATH = os.path.join('/content', 'drive', 'MyDrive', GOOGLE_DRIVE_PATH_POST_MYDRIVE)
print(os.listdir(GOOGLE_DRIVE_PATH))

os.chdir(GOOGLE_DRIVE_PATH)
sys.path.append(GOOGLE_DRIVE_PATH)

['LSTM.ipynb', '.gitignore', '.pre-commit-config.yaml', '.python-version', 'README.md', 'pyproject.toml', 'uv.lock', '.kaggle', '.venv', 'dataset', '.git', '.idea', 'epoch_10_weights.pth', '__pycache__', 'models', '__init__.py.txt', 'utils_train.py', 'best_model_bert.pt', 'custom_model_lstm.pth', 'test_predictions_LSTM.csv', 'custom_model_lstm_2.pth', 'test_predictions_LSTM_with_3layers.csv', 'custom_model_bilstm.pth', 'test_predictions_bi_LSTM.csv']


In [4]:
# if running locally set GOOGLE PATH
import sys
if 'google.colab' in sys.modules:
  print(f'Running in google colab. Our path is `{GOOGLE_DRIVE_PATH}`')
else:
  GOOGLE_DRIVE_PATH = '.'
  print('Running locally.')

Running in google colab. Our path is `/content/drive/MyDrive/OMSCS/DeepLearning/main_repo`


In [5]:
# dataset.py
import re
import nltk
import numpy as np
import pandas as pd
import torch
from nltk.tokenize import word_tokenize
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import random

nltk.download('punkt')
nltk.download('punkt_tab') # Download punkt_tab resource

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

def preprocess_and_tokenize(df, max_len):
    df['comment_text'] = df['comment_text'].apply(clean_text)
    df['tokens'] = df['comment_text'].apply(word_tokenize)

    all_tokens = [tok for toks in df['tokens'] for tok in toks]
    vocab = {word: i+2 for i, (word, _) in enumerate(Counter(all_tokens).items())}
    vocab['<PAD>'] = 0
    vocab['<UNK>'] = 1

    def encode(tokens):
        return [vocab.get(token, vocab['<UNK>']) for token in tokens[:max_len]]

    def pad(seq):
        return seq + [0] * (max_len - len(seq))

    df['input_ids'] = df['tokens'].apply(lambda t: pad(encode(t)))
    return df, vocab

def load_glove_embeddings(glove_path, vocab, embedding_dim):
    embeddings = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            vals = line.split()
            word = vals[0]
            vec = np.asarray(vals[1:], dtype='float32')
            embeddings[word] = vec

    matrix = np.zeros((len(vocab), embedding_dim))
    for word, idx in vocab.items():
        if word in embeddings:
            matrix[idx] = embeddings[word]
    return matrix

class ToxicDataset(Dataset):
    def __init__(self, input_ids, labels):
        self.input_ids = torch.tensor(input_ids, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.labels[idx]


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [6]:
# model.py
import torch
import torch.nn as nn

class ToxicCommentClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, embedding_matrix, hidden_size=64, output_dim=6,dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float32))
        self.embedding.weight.requires_grad = False
        self.lstm = nn.LSTM(embedding_dim, hidden_size,num_layers=3,bidirectional=True,batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size*2, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        h_forward = h_n[0]  # forward direction
        h_backward = h_n[1]  # backward direction
        h_cat = torch.cat((h_forward, h_backward), dim=1)  # [batch, hidden*2]
        out = self.fc(self.dropout(h_cat))
        return out


In [15]:
class CBFocalLoss(nn.Module):
    def __init__(self, samples_per_cls, beta=0.9999, gamma=2.0):
        super(CBFocalLoss, self).__init__()
        self.beta = beta
        self.gamma = gamma

        effective_num = 1.0 - torch.pow(self.beta, samples_per_cls)
        weights = (1.0 - self.beta) / (effective_num + 1e-8)
        weights = weights / weights.sum() * len(samples_per_cls)  # Normalize

        self.class_weights = weights.to(torch.float32)

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = torch.clamp(probs, 1e-4, 1 - 1e-4)

        # 👇 Move weights to same device as logits
        weights = self.class_weights.to(logits.device).unsqueeze(0)

        focal_weight = torch.pow(1.0 - probs * targets - (1 - probs) * (1 - targets), self.gamma)

        bce = - (targets * torch.log(probs) + (1 - targets) * torch.log(1 - probs))
        loss = weights * focal_weight * bce

        return loss.mean()

In [8]:
# train.py
import torch
from sklearn.metrics import f1_score

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            preds = torch.sigmoid(logits).cpu().numpy() > 0.5
            all_preds.extend(preds)
            all_labels.extend(y.numpy())
    f1 = f1_score(all_labels, all_preds, average='macro')
    return f1


In [9]:
class Config:
    EMBEDDING_DIM = 100
    HIDDEN_SIZE = 64
    MAX_LEN = 100
    BATCH_SIZE = 32
    NUM_EPOCHS = 10
    LEARNING_RATE = 1e-3
    EMBEDDING_PATH = f"dataset/word_embeddings/glove.6B/glove.6B.100d.txt"
    MODEL_PATH = f"bilstm.pth"
    DATA_PATH = f"dataset/.cache/kagglehub/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip"
    LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
    SEED = 42

In [10]:
from sklearn.model_selection import train_test_split

cfg = Config()
set_seed(Config.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load and process data
df = pd.read_csv(cfg.DATA_PATH)
df, vocab = preprocess_and_tokenize(df, cfg.MAX_LEN)
embedding_matrix = load_glove_embeddings(cfg.EMBEDDING_PATH, vocab, cfg.EMBEDDING_DIM)
samples_per_cls = torch.tensor([df[col].sum() for col in cfg.LABELS], dtype=torch.float32)

In [11]:
samples_per_cls

tensor([15294.,  1595.,  8449.,   478.,  7877.,  1405.])

In [12]:
X_train, X_val, y_train, y_val = train_test_split(
    df['input_ids'].tolist(),
    df[cfg.LABELS].values.tolist(),
    test_size=0.2
)

train_ds = ToxicDataset(X_train, y_train)
val_ds = ToxicDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE)

In [18]:
# Model setup
model = ToxicCommentClassifier(len(vocab), cfg.EMBEDDING_DIM, embedding_matrix,
                               hidden_size=cfg.HIDDEN_SIZE,
                               output_dim=len(cfg.LABELS)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE)
# Scheduler (track validation F1 to reduce LR when F1 plateaus)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2, verbose=True
)
# loss_fn = torch.nn.BCEWithLogitsLoss()
criterion = CBFocalLoss(samples_per_cls=samples_per_cls, beta=0.9999, gamma=2.0)

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [19]:
# Training loop
for epoch in range(20):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_f1 = evaluate(model, val_loader, device)
    # 🔥 Step the scheduler
    scheduler.step(val_f1)
    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val F1: {val_f1:.4f}")

Epoch 1 | Loss: 0.0104 | Val F1: 0.3440
Epoch 2 | Loss: 0.0079 | Val F1: 0.4405
Epoch 3 | Loss: 0.0072 | Val F1: 0.4603
Epoch 4 | Loss: 0.0066 | Val F1: 0.4991
Epoch 5 | Loss: 0.0063 | Val F1: 0.4837
Epoch 6 | Loss: 0.0059 | Val F1: 0.5172
Epoch 7 | Loss: 0.0058 | Val F1: 0.4962
Epoch 8 | Loss: 0.0055 | Val F1: 0.5422
Epoch 9 | Loss: 0.0052 | Val F1: 0.5335
Epoch 10 | Loss: 0.0051 | Val F1: 0.5493
Epoch 11 | Loss: 0.0049 | Val F1: 0.5683
Epoch 12 | Loss: 0.0047 | Val F1: 0.5663
Epoch 13 | Loss: 0.0046 | Val F1: 0.5469
Epoch 14 | Loss: 0.0045 | Val F1: 0.5406
Epoch 15 | Loss: 0.0042 | Val F1: 0.5700
Epoch 16 | Loss: 0.0040 | Val F1: 0.5700
Epoch 17 | Loss: 0.0039 | Val F1: 0.5737
Epoch 18 | Loss: 0.0038 | Val F1: 0.5618
Epoch 19 | Loss: 0.0038 | Val F1: 0.5801
Epoch 20 | Loss: 0.0037 | Val F1: 0.5647


In [20]:
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'vocab': vocab,
},'custom_model_bilstm_with_bifocal_loss.pth')

In [21]:
# inference.py

import torch
import pandas as pd
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import classification_report

def prepare_test_data(df, vocab, max_len):
    df['comment_text'] = df['comment_text'].apply(clean_text)
    df['tokens'] = df['comment_text'].apply(word_tokenize)

    def encode(tokens):
        return [vocab.get(tok, vocab['<UNK>']) for tok in tokens[:max_len]]

    def pad(seq):
        return seq + [0] * (max_len - len(seq))

    df['input_ids'] = df['tokens'].apply(lambda x: pad(encode(x)))
    return df

def run_inference(model, test_loader, device):
    model.eval()
    predictions = []

    with torch.no_grad():
        for inputs, _ in test_loader:  # labels are dummy here
            inputs = inputs.to(device)
            logits = model(inputs)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            predictions.extend(preds)

    return np.array(predictions)


In [23]:
# Load test data
test_df = pd.read_csv("dataset/.cache/kagglehub/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip")
test_labels_df = pd.read_csv("dataset/.cache/kagglehub/competitions/jigsaw-toxic-comment-classification-challenge/test_labels.csv.zip")  # Optional

# Prepare
test_df = prepare_test_data(test_df, vocab, cfg.MAX_LEN)
dummy_labels = [[0]*len(cfg.LABELS)] * len(test_df)  # dummy labels to match dataset format
test_ds = ToxicDataset(test_df['input_ids'].tolist(), dummy_labels)
test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE)

# Load trained model
checkpoint = torch.load('custom_model_bilstm.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Run inference
preds = run_inference(model, test_loader, device)

# Save predictions
output_df = test_df.copy()
output_df[cfg.LABELS] = preds
output_df.to_csv('test_predictions_bi_LSTM_with_bifocal_loss.csv', index=False)

In [24]:
from sklearn.metrics import classification_report

true_labels = test_labels_df[cfg.LABELS].values
mask = (true_labels < 0).any(axis=1)
true_labels_copy = true_labels[~mask]
pred_labels = preds[~mask]

print(classification_report(true_labels_copy, pred_labels, target_names=cfg.LABELS, zero_division=0))

               precision    recall  f1-score   support

        toxic       0.54      0.81      0.65      6090
 severe_toxic       0.30      0.39      0.34       367
      obscene       0.59      0.72      0.65      3691
       threat       0.47      0.36      0.40       211
       insult       0.57      0.63      0.60      3427
identity_hate       0.48      0.40      0.44       712

    micro avg       0.55      0.71      0.62     14498
    macro avg       0.49      0.55      0.51     14498
 weighted avg       0.55      0.71      0.61     14498
  samples avg       0.07      0.07      0.06     14498



(6,)